<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio 1, Modulo 2 Tema 2</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>

Este cuaderno está diseñado desde un enfoque académico diseñado para poner en práctica los conceptos explicados en el Tema 2: Programación de Apliacaciones Spark del Modulo 2 de la asignatura.


## Objetivos generales del cuaderno

A lo largo de este notebook se persiguen los siguientes objetivos:

1. Cargar y comprender la estructura real del dataset.
2. Aplicar procesos de preparación y tipado de datos en Spark.
3. Enriquecer la tabla principal con dimensiones auxiliares.
4. Construir análisis agregados de interés para negocio.
5. Aplicar técnicas avanzadas de Spark:
   - `groupBy`
   - `pivot`
   - funciones de ventana
   - `broadcast join`
   - `left_semi` y `left_anti`
   - joins por rango temporal
6. Relacionar los resultados obtenidos con un contexto real de analítica de ventas.

## Dataset utilizado

Este notebook está adaptado a los tres ficheros reales del dataset de Kaggle:

- `sales.csv`
- `product_hierarchy.csv`
- `store_cities.csv`

## Columnas principales

### `sales.csv`
- `store_id`
- `product_id`
- `date`
- `sales`
- `revenue`
- `stock`
- `price`
- `promo_type_1`
- `promo_bin_1`
- `promo_type_2`
- `promo_bin_2`
- `promo_discount_2`
- `promo_discount_type_2`

### `product_hierarchy.csv`
- `product_id`
- `product_length`
- `product_depth`
- `product_width`
- `hierarchy1_id`
- `hierarchy2_id`
- `hierarchy3_id`
- `hierarchy4_id`
- `hierarchy5_id`

### `store_cities.csv`
- `store_id`
- `storetype_id`
- `store_size`
- `city_id`

### Variable adicional relevante
- `train_or_test`: etiqueta de partición para entrenamiento y prueba.

## Recomendación de uso

Se recomienda ejecutar el cuaderno **bloque a bloque**, revisando siempre:
- el esquema resultante,
- las primeras filas,
- y la interpretación analítica de cada salida.


## 1. Creación de la sesión de Spark

En cualquier aplicación PySpark, el punto de entrada es la `SparkSession`.
Esta sesión representa la conexión lógica con el motor de Spark y permite leer datos, transformarlos y ejecutar acciones.

En un contexto académico, este paso es importante porque muestra el inicio del pipeline analítico y el arranque del entorno distribuido.




In [ ]:
!pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Retail_Sales_Analysis")
    .getOrCreate()
)

spark

### Ejercicio propuesto

1. Explica con tus palabras que papel desempena `SparkSession` dentro de una aplicacion PySpark.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

`SparkSession` es el punto de entrada principal de una aplicacion PySpark. Permite crear DataFrames, leer datos, ejecutar SQL, acceder a configuracion de Spark y coordinar el trabajo con el motor distribuido.

</details>

2. Indica que diferencias conceptuales existen entre `SparkSession` y `SparkContext`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

`SparkContext` representa la conexion de bajo nivel con el cluster y gestiona recursos y RDDs. `SparkSession` es una API de nivel superior que integra `SparkContext`, SQL, DataFrames, catalogo y lectura/escritura de datos. En aplicaciones modernas se suele trabajar con `SparkSession`.

</details>

3. Comprueba en tu entorno que version de Spark estas ejecutando.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Se puede comprobar con:

```python
spark.version
```

Tambien puede imprimirse junto con informacion del contexto:

```python
print(spark.version)
print(spark.sparkContext.appName)
```

</details>


## 2. Definición de rutas de entrada

En este bloque se definen las rutas de los tres ficheros del dataset.



In [ ]:
sales_path = "../Tema4/sales.csv"
product_path = "DatasetTema2/product_hierarchy.csv"
stores_path = "DatasetTema2/store_cities.csv"

### Ejercicio propuesto

1. Modifica las rutas para adaptarlas a tu entorno local o al entorno Docker de Spark.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

La ruta debe apuntar al lugar donde estan los ficheros accesibles desde el proceso de Spark. En este proyecto, para `sales.csv` se usa:

```python
sales_path = "../Tema4/sales.csv"
```

Si se ejecuta desde Docker, la ruta debe coincidir con el volumen montado, por ejemplo `/opt/spark/data/sales.csv`.

</details>

2. Explica por que es recomendable no hardcodear rutas complejas en multiples celdas del notebook.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Porque dificulta mover el notebook entre entornos, aumenta el riesgo de inconsistencias y obliga a modificar varias celdas si cambia la ubicacion de los datos. Es mejor centralizar rutas en una unica celda de configuracion o usar variables.

</details>


## 3. Carga de los ficheros CSV

En este bloque se cargan los tres datasets.
Se utiliza `header=True` para interpretar correctamente la cabecera y `inferSchema=True` para que Spark intente deducir los tipos de datos automáticamente.

Aunque `inferSchema` es muy útil para exploración, en proyectos más robustos suele ser recomendable definir esquemas explícitos para mejorar:
- la reproducibilidad,
- el rendimiento de lectura,
- y el control sobre los tipos.


In [ ]:
sales_df = spark.read.csv(sales_path, header=True, inferSchema=True)
product_df = spark.read.csv(product_path, header=True, inferSchema=True)
stores_df = spark.read.csv(stores_path, header=True, inferSchema=True)

print("Esquema de sales.csv")
sales_df.printSchema()

print("Esquema de product_hierarchy.csv")
product_df.printSchema()

print("Esquema de store_cities.csv")
stores_df.printSchema()

### Ejercicio propuesto

1. Revisa los esquemas obtenidos y anota que columnas Spark ha interpretado como numericas y cuales como texto.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

En `sales.csv`, columnas como `sales`, `revenue`, `stock`, `price` y `promo_discount_2` deberian interpretarse como numericas. Identificadores como `product_id`, `store_id`, `promo_type_1`, `promo_bin_1`, `promo_type_2` y `promo_bin_2` se tratan como texto. `date` puede aparecer como texto si no se convierte explicitamente a tipo fecha.

</details>

2. Indica que riesgos puede tener depender unicamente de `inferSchema=True`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

`inferSchema=True` puede inferir tipos incorrectos si la muestra no representa todos los datos, puede convertir fechas o codigos de forma no deseada y anade coste de lectura porque Spark debe inspeccionar el fichero. En pipelines reproducibles es preferible declarar un esquema explicito.

</details>

3. Elabora una breve propuesta de esquema explicito para `sales.csv`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Una propuesta razonable es:

```python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

sales_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("sales", DoubleType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("stock", DoubleType(), True),
    StructField("price", DoubleType(), True),
    StructField("promo_type_1", StringType(), True),
    StructField("promo_bin_1", StringType(), True),
    StructField("promo_type_2", StringType(), True),
    StructField("promo_bin_2", StringType(), True),
    StructField("promo_discount_2", DoubleType(), True),
    StructField("promo_discount_type_2", StringType(), True),
])
```

</details>


## 4. Exploración inicial de los datos

Antes de transformar un dataset es fundamental inspeccionar algunas filas.
Esta práctica permite:
- detectar valores nulos,
- entender la granularidad,
- validar nombres de columnas,
- y confirmar si la estructura coincide con la documentación del dataset.


In [ ]:
sales_df.show(5, truncate=False)
product_df.show(5, truncate=False)
stores_df.show(5, truncate=False)

### Ejercicio propuesto

1. Describe que representa una fila de `sales.csv`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Una fila representa la observacion de un producto concreto en una tienda concreta para una fecha concreta, con informacion de ventas, ingresos, stock, precio y promociones aplicadas.

</details>

2. Indica si la granularidad del dataset parece ser diaria, semanal o transaccional.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

La granularidad parece diaria, porque existe una columna `date` y cada registro resume el comportamiento de una combinacion producto-tienda en un dia. No parece transaccional porque no hay identificador de ticket, cliente u operacion individual.

</details>

3. Que columnas del dataset consideras hechos y cuales dimensiones?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Hechos o metricas: `sales`, `revenue`, `stock`, `price`, `promo_discount_2`. Dimensiones o atributos: `product_id`, `store_id`, `date`, `promo_type_1`, `promo_bin_1`, `promo_type_2`, `promo_bin_2`, `promo_discount_type_2`.

</details>


## 5. Tipado y preparación de `sales.csv`

Este bloque transforma `sales_df` para preparar los datos antes del análisis.

### Transformaciones realizadas

- Conversión de `date` al tipo `date`.
- Conversión explícita de variables numéricas.
- Creación de columnas temporales (`year`, `month`, `weekofyear`).
- Generación de indicadores auxiliares:
  - `is_stockout`: marca si el stock final del día es cero o negativo.
  - `avg_ticket_est`: estimación del ingreso medio por unidad vendida.

Este tipo de preparación es una etapa clásica de ingeniería de datos y análisis exploratorio.


In [ ]:
sales_df = (
    sales_df
    .withColumn("date", F.to_date("date"))
    .withColumn("sales", F.col("sales").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("stock", F.col("stock").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("promo_discount_2", F.col("promo_discount_2").cast("double"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("weekofyear", F.weekofyear("date"))
    .withColumn("is_stockout", F.when(F.col("stock") <= 0, 1).otherwise(0))
    .withColumn("avg_ticket_est", F.when(F.col("sales") != 0, F.col("revenue") / F.col("sales")))
)

sales_df.printSchema()
sales_df.show(5, truncate=False)

### Comentario analítico

La variable `avg_ticket_est` no representa un ticket por cliente, ya que el dataset no incluye clientes individuales.
Sin embargo, sí puede interpretarse como una **aproximación al ingreso medio por unidad vendida** en cada combinación tienda-producto-fecha.

La variable `is_stockout` será especialmente útil para analizar roturas de stock, un aspecto muy relevante en entornos retail.


### Ejercicio propuesto

1. Explica la diferencia entre convertir `date` a tipo `date` y dejarla como texto.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Como texto, `date` solo permite comparaciones lexicograficas y manipulaciones de cadenas. Como tipo `date`, Spark puede ordenar temporalmente, extraer ano o mes, aplicar ventanas temporales y realizar filtros por rango de fechas de forma correcta.

</details>

2. Justifica por que se crea la columna `is_stockout`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

`is_stockout` convierte una condicion de negocio, stock igual o inferior a cero, en una variable analitica. Facilita calcular ratios de rotura, agrupar por producto o tienda y detectar problemas de disponibilidad.

</details>

3. Propon dos columnas derivadas adicionales que podrian ser utiles para el analisis.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Dos ejemplos son:

```python
sales_df = sales_df.withColumn("year_month", F.date_format("date", "yyyy-MM"))
sales_df = sales_df.withColumn("has_promo", F.when(F.col("promo_type_1").isNotNull() | F.col("promo_type_2").isNotNull(), 1).otherwise(0))
```

`year_month` permite agregaciones mensuales y `has_promo` permite comparar ventas con y sin promocion.

</details>


## 6. Enriquecimiento con dimensiones de producto y tienda

Una práctica habitual en analítica y Data Warehousing es trabajar con una tabla de hechos y varias dimensiones.

En este caso:
- `sales_df` actúa como tabla principal de hechos.
- `product_df` aporta jerarquía y dimensiones físicas del producto.
- `stores_df` aporta características de la tienda.

La unión se realiza con `broadcast`, una optimización especialmente útil cuando las tablas dimensionales son lo bastante pequeñas como para replicarse en memoria en los ejecutores.


In [ ]:
retail_df = (
    sales_df
    .join(F.broadcast(product_df), "product_id", "left")
    .join(F.broadcast(stores_df), "store_id", "left")
)

retail_df.printSchema()
retail_df.show(5, truncate=False)

### Comentario analítico

Gracias a este enriquecimiento, el análisis deja de estar limitado a tienda, producto y fecha, y pasa a incorporar:
- jerarquías de producto,
- tamaño de tienda,
- tipo de tienda,
- ciudad.

Esto permite formular preguntas mucho más ricas, por ejemplo:
- ¿qué jerarquías venden más?
- ¿qué tamaños de tienda presentan más roturas de stock?
- ¿hay diferencias entre ciudades?


### Ejercicio propuesto

1. Explica por que este patron de union se parece a un modelo estrella.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

La tabla `sales` actua como tabla de hechos, porque contiene medidas como ventas e ingresos. Las tablas de productos y tiendas actuan como dimensiones que enriquecen esos hechos con atributos descriptivos. Ese patron es propio de un modelo estrella.

</details>

2. Que ventaja tiene usar `broadcast` en este caso?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Si las tablas de productos y tiendas son pequenas, `broadcast` evita un shuffle grande: Spark envia la dimension completa a cada executor y puede hacer el join localmente contra la tabla de hechos.

</details>

3. Que ocurriria si se hiciera `broadcast` sobre una tabla demasiado grande?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Podria saturarse la memoria de los executors o del driver, aumentar la presion de GC y provocar errores `OutOfMemoryError`. El broadcast solo es recomendable para tablas pequenas o moderadas.

</details>


## 7. Comprobación de columnas tras el enriquecimiento

En datasets reales es recomendable revisar las columnas finales antes de continuar.
Este paso ayuda a:
- validar que las uniones se han realizado correctamente,
- comprobar que no hay colisiones inesperadas,
- y confirmar que las columnas necesarias están disponibles.


In [ ]:
retail_df.columns

### Ejercicio propuesto

1. Localiza que columnas pertenecen a ventas, cuales a productos y cuales a tiendas.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Ventas: `product_id`, `store_id`, `date`, `sales`, `revenue`, `stock`, `price` y columnas de promocion. Productos: atributos como `hierarchy1_id`, `hierarchy2_id`, dimensiones fisicas del producto y niveles de jerarquia. Tiendas: `storetype_id`, `store_size`, `city_id` y otros atributos de tienda disponibles.

</details>

2. Clasifica las columnas en metricas, identificadores y atributos descriptivos.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Metricas: `sales`, `revenue`, `stock`, `price`, ratios o medias derivadas. Identificadores: `product_id`, `store_id`, `city_id`, jerarquias y `campaign_id` si existe. Atributos descriptivos: tipo de tienda, tamano, categorias de producto, tipos de promocion y bins promocionales.

</details>


## 8. KPI inicial: ventas y facturación por tienda

Una primera vista útil consiste en agregar por `store_id` para obtener indicadores agregados de rendimiento.

### Métricas calculadas

- `total_sales_qty`: suma de unidades vendidas.
- `total_revenue`: suma de ingresos.
- `avg_price`: precio medio.
- `avg_stock`: stock medio final.

Este tipo de tabla permite identificar tiendas con alto volumen, alto ingreso o posibles diferencias operativas.


In [ ]:
kpi_store = (
    retail_df
    .groupBy("store_id")
    .agg(
        F.sum("sales").alias("total_sales_qty"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("price").alias("avg_price"),
        F.avg("stock").alias("avg_stock")
    )
    .orderBy(F.desc("total_revenue"))
)

kpi_store.show(20, truncate=False)

### Ejercicio propuesto

1. Identifica las tiendas con mayor facturacion.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Se identifican ordenando el KPI por `total_revenue` descendente:

```python
kpi_store.orderBy(F.desc("total_revenue")).show(10)
```

Las primeras filas son las tiendas con mayor facturacion total en el periodo analizado.

</details>

2. Existe relacion aparente entre `avg_stock` y `total_revenue`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Puede existir relacion si las tiendas con mayor stock tambien venden mas, pero no debe asumirse causalidad. Un stock alto puede reflejar mayor demanda esperada, mayor tamano de tienda o menor rotacion. Conviene analizar correlacion y segmentar por tipo o tamano de tienda.

</details>

3. Propon una visualizacion adecuada para representar este resultado en el TFG.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Un grafico de barras horizontales con las 10 tiendas de mayor `total_revenue` es adecuado para ranking. Para comparar stock e ingresos, un scatter plot con `avg_stock` en el eje X y `total_revenue` en el eje Y permitiria observar relacion o dispersion.

</details>


## 9. Análisis por ciudad, tipo y tamaño de tienda

En este bloque se amplía la agregación anterior utilizando atributos de la dimensión de tienda:
- `city_id`
- `storetype_id`
- `store_size`

Esto permite estudiar el comportamiento agregado por perfil de tienda y contexto geográfico.


In [ ]:
kpi_city_storetype = (
    retail_df
    .groupBy("city_id", "storetype_id", "store_size")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total"),
        F.avg("price").alias("avg_price")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_city_storetype.show(20, truncate=False)

### Ejercicio propuesto

1. Senala que combinacion de ciudad, tipo y tamano parece mas rentable.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Debe observarse la salida ordenada por `total_revenue` descendente. La combinacion de `city_id`, `storetype_id` y `store_size` situada en la primera fila es la mas rentable segun la facturacion agregada del periodo.

</details>

2. Que utilidad tendria este analisis para un responsable comercial?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Permite detectar formatos de tienda y ubicaciones con mejor rendimiento, priorizar inversiones, ajustar surtido, revisar promociones y comparar desempeno entre ciudades o tipos de establecimiento.

</details>

3. Que limitaciones tiene este resultado si `city_id` no esta descrito semanticamente?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Si `city_id` es solo un codigo anonimo, no se puede interpretar geograficamente ni vincular con variables externas como poblacion, renta o competencia. El analisis sirve para comparar codigos, pero no para explicar contexto territorial.

</details>


## 10. Tabla dinámica (pivot) por tipo de promoción


El objetivo es construir una tabla donde:
- las filas sean las tiendas,
- las columnas sean los tipos de promoción,
- y el valor agregado sea la suma de `revenue`.

Este patrón es útil para comparar el impacto económico de distintos tipos promocionales.


In [ ]:
promo_type_1_values = [r[0] for r in retail_df.select("promo_type_1").distinct().orderBy("promo_type_1").collect() if r[0] is not None]
promo_type_1_values[:20]

In [ ]:
df_pivot_promo1 = (
    retail_df
    .groupBy("store_id")
    .pivot("promo_type_1", promo_type_1_values)
    .agg(F.sum("revenue"))
)

df_pivot_promo1.show(20, truncate=False)

### Comentario analítico

Las tablas pivot son especialmente útiles para:
- reporting,
- dashboards,
- comparación entre grupos,
- y exportación a herramientas de visualización.

No obstante, pueden ser costosas si la cardinalidad de la columna pivotada es alta.


### Ejercicio propuesto

1. Explica que diferencias hay entre un `groupBy` tradicional y un `pivot`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

`groupBy` genera filas agrupadas por claves y metricas. `pivot` transforma valores de una columna en columnas nuevas, creando una tabla mas matricial. Es util para comparar categorias, aunque puede generar muchas columnas si la cardinalidad es alta.

</details>

2. Repite el analisis usando `promo_type_2`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Un ejemplo seria:

```python
kpi_promo2 = (
    retail_df
    .groupBy("store_id")
    .pivot("promo_type_2")
    .agg(F.sum("revenue"))
)

kpi_promo2.show()
```

</details>

3. Sustituye la suma de `revenue` por la suma de `sales` y compara los resultados.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Se puede calcular asi:

```python
kpi_promo_units = (
    retail_df
    .groupBy("store_id")
    .pivot("promo_type_1")
    .agg(F.sum("sales"))
)
```

`revenue` mide facturacion monetaria, mientras que `sales` mide unidades vendidas. Un producto barato puede liderar en unidades pero no en ingresos.

</details>


## 11. Top 3 productos por jerarquía principal

En lugar de utilizar una categoría genérica, aquí se aprovecha la jerarquía real de producto mediante `hierarchy1_id`.

El procedimiento consiste en:
1. agregar ventas por `hierarchy1_id` y `product_id`,
2. ordenar por facturación,
3. asignar un ranking con `row_number()`,
4. filtrar el Top 3 por grupo.

Este patrón es uno de los usos clásicos de las funciones de ventana.


In [ ]:
df_prod_h1 = (
    retail_df
    .groupBy("hierarchy1_id", "product_id")
    .agg(
        F.sum("revenue").alias("revenue_total"),
        F.sum("sales").alias("sales_qty")
    )
)

w_h1 = Window.partitionBy("hierarchy1_id").orderBy(F.desc("revenue_total"))

df_top_h1 = (
    df_prod_h1
    .withColumn("rn", F.row_number().over(w_h1))
    .filter(F.col("rn") <= 3)
    .drop("rn")
    .orderBy("hierarchy1_id", F.desc("revenue_total"))
)

df_top_h1.show(50, truncate=False)

### Ejercicio propuesto

1. Explica por que se utiliza `partitionBy("hierarchy1_id")`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Se usa para calcular el ranking de productos dentro de cada grupo de la jerarquia principal. Asi, el `row_number` se reinicia para cada `hierarchy1_id` y se obtienen tops comparables por categoria.

</details>

2. Que cambiaria si se usara `rank()` en lugar de `row_number()`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

`row_number()` asigna posiciones unicas aunque haya empates. `rank()` asigna la misma posicion a registros empatados y deja saltos en la numeracion. Si dos productos empatan, ambos podrian tener rank 1 y el siguiente rank seria 3.

</details>

3. Repite el ejercicio con `hierarchy2_id`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

La ventana se cambiaria a:

```python
w = Window.partitionBy("hierarchy2_id").orderBy(F.desc("total_revenue"))
```

Despues se aplica la misma logica de ranking y filtro sobre la nueva jerarquia.

</details>


## 12. Top 3 productos por tienda

Este análisis cambia la perspectiva: ahora no interesa la jerarquía global, sino el rendimiento de producto dentro de cada tienda.

Este tipo de ranking puede ser útil para:
- gestión del surtido,
- optimización de inventario,
- detección de productos estrella por ubicación.


In [ ]:
df_prod_store = (
    retail_df
    .groupBy("store_id", "product_id")
    .agg(
        F.sum("revenue").alias("revenue_total"),
        F.sum("sales").alias("sales_qty")
    )
)

w_store = Window.partitionBy("store_id").orderBy(F.desc("revenue_total"))

df_top_store = (
    df_prod_store
    .withColumn("rn", F.row_number().over(w_store))
    .filter(F.col("rn") <= 3)
    .drop("rn")
)

df_top_store.show(50, truncate=False)

### Ejercicio propuesto

1. Identifica si los productos top se repiten entre tiendas o si existe heterogeneidad.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Se puede revisar agrupando los productos top por `product_id` y contando en cuantas tiendas aparecen. Si pocos productos aparecen en muchas tiendas, hay repeticion. Si cada tienda tiene productos top distintos, hay heterogeneidad.

</details>

2. Propon una interpretacion comercial para este resultado.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Si los productos top se repiten, puede existir un surtido estrella comun que conviene asegurar. Si hay heterogeneidad, las preferencias pueden depender de la tienda, ciudad, tamano o perfil de cliente, y el surtido deberia adaptarse localmente.

</details>

3. Anade el `store_size` al resultado final para enriquecer el analisis.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Si `store_size` esta en `retail_df`, se puede incluir en el agrupado o recuperarlo despues con un join:

```python
df_prod_store_size = df_prod_store.join(
    stores_df.select("store_id", "store_size").dropDuplicates(),
    on="store_id",
    how="left"
)
```

</details>


## 13. Media móvil de 7 días por tienda y producto

Dado que el dataset tiene granularidad diaria, una ventana temporal móvil es muy adecuada.

En este caso se calcula la media de `revenue` en los últimos 7 días para cada combinación de:
- `store_id`
- `product_id`

Este tipo de métrica ayuda a suavizar fluctuaciones diarias y detectar tendencias recientes.


In [ ]:
w7d = (
    Window
    .partitionBy("store_id", "product_id")
    .orderBy(F.col("date").cast("timestamp").cast("long"))
    .rangeBetween(-7 * 24 * 3600, 0)
)

df_roll_7d = retail_df.withColumn("revenue_avg_7d", F.avg("revenue").over(w7d))

df_roll_7d.select(
    "store_id", "product_id", "date", "revenue", "revenue_avg_7d"
).show(30, truncate=False)

### Comentario analítico

La media móvil permite reducir ruido y comparar el comportamiento actual con una tendencia reciente.
Es especialmente útil en retail para:
- seguimiento de producto,
- detección de cambios de demanda,
- análisis del efecto de promociones.


### Ejercicio propuesto

1. Explica la diferencia entre `rangeBetween` y `rowsBetween`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

`rowsBetween` define la ventana por numero de filas relativas. `rangeBetween` define la ventana por rango de valores de la columna de ordenacion. Para fechas convertidas a timestamp, `rangeBetween` permite ventanas temporales reales, por ejemplo ultimos 7 dias.

</details>

2. Justifica por que en este caso se usa una ventana temporal y no una ventana por numero de filas.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Porque el objetivo es calcular una media movil basada en dias, no en un numero fijo de registros. Si faltan dias o hay diferente numero de filas por producto, una ventana por filas no representaria necesariamente 7 dias reales.

</details>

3. Calcula tambien una media movil de 14 dias.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Ejemplo:

```python
seconds_14d = 14 * 24 * 60 * 60
w14 = Window.partitionBy("product_id", "store_id").orderBy(F.col("date_ts")).rangeBetween(-seconds_14d, 0)

df_ma = df_ma.withColumn("revenue_avg_14d", F.avg("revenue").over(w14))
```

</details>


## 14. Media móvil de 30 días

Una ventana más amplia permite observar tendencias menos sensibles al ruido diario.

Comparar la media móvil de 7 días con la de 30 días puede resultar útil para detectar:
- aceleraciones,
- desaceleraciones,
- o comportamientos anómalos.


In [ ]:
w30d = (
    Window
    .partitionBy("store_id", "product_id")
    .orderBy(F.col("date").cast("timestamp").cast("long"))
    .rangeBetween(-30 * 24 * 3600, 0)
)

df_roll_30d = retail_df.withColumn("revenue_avg_30d", F.avg("revenue").over(w30d))

df_roll_30d.select(
    "store_id", "product_id", "date", "revenue", "revenue_avg_30d"
).show(30, truncate=False)

### Ejercicio propuesto

1. Compara conceptualmente las ventanas de 7 y 30 dias.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

La ventana de 7 dias es mas sensible a cambios recientes y sirve para seguimiento operativo. La de 30 dias suaviza mas el ruido y refleja una tendencia mensual mas estable.

</details>

2. Cual usarias para seguimiento operativo diario y cual para reporting mensual?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Para seguimiento operativo diario usaria 7 dias, porque detecta cambios rapidos. Para reporting mensual usaria 30 dias, porque ofrece una vision mas estable y menos influida por variaciones puntuales.

</details>

3. Calcula una columna adicional con la diferencia entre `revenue` y `revenue_avg_30d`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Ejemplo:

```python
df_ma = df_ma.withColumn(
    "revenue_vs_avg_30d",
    F.col("revenue") - F.col("revenue_avg_30d")
)
```

Un valor positivo indica ingresos por encima de la media movil de 30 dias.

</details>


## 15. Análisis de roturas de stock

Una rotura de stock implica que al final del día el producto queda sin inventario disponible.
Esto puede ser indicio de:
- alta demanda,
- mala planificación,
- o problemas de reposición.

En este bloque se calcula:
- el número de días con rotura de stock,
- el número total de días observados,
- y el ratio de rotura por combinación tienda-producto.


In [ ]:
stockout_kpi = (
    retail_df
    .groupBy("store_id", "product_id")
    .agg(
        F.sum("is_stockout").alias("days_stockout"),
        F.count("*").alias("days_total"),
        (F.sum("is_stockout") / F.count("*")).alias("stockout_ratio")
    )
    .orderBy(F.desc("stockout_ratio"))
)

stockout_kpi.show(30, truncate=False)

### Ejercicio propuesto

1. Explica por que el `stockout_ratio` puede ser mas informativo que el numero bruto de dias.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

El numero bruto de dias no considera cuantos dias se observo cada producto o tienda. El ratio normaliza los dias de rotura sobre el total observado, permitiendo comparaciones mas justas entre productos con distinta cobertura temporal.

</details>

2. Que medidas podria tomar una empresa ante productos con un ratio elevado?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Podria revisar niveles de reposicion, aumentar stock de seguridad, mejorar prediccion de demanda, ajustar frecuencia de suministro o investigar si hay problemas logisticos recurrentes.

</details>

3. Enlaza este analisis con la gestion de inventario.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

El analisis de roturas conecta ventas perdidas, disponibilidad y planificacion de stock. Un ratio alto puede indicar que la empresa no esta capturando toda la demanda potencial y necesita optimizar inventario y reposicion.

</details>


## 16. Días con ventas y stock final nulo

Una situación especialmente interesante es aquella en la que hay ventas durante el día pero el stock final es cero o negativo.

Este patrón puede reflejar:
- fuerte rotación,
- agotamiento del inventario,
- o necesidad de revisar el aprovisionamiento.


In [ ]:
df_sales_stock_alert = retail_df.filter(
    (F.col("sales") > 0) & (F.col("stock") <= 0)
)

df_sales_stock_alert.select(
    "store_id", "product_id", "date", "sales", "revenue", "stock"
).show(30, truncate=False)

### Ejercicio propuesto

1. Interpreta el significado de una fila devuelta por este filtro.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Una fila devuelta representa un caso donde hay ventas o ingresos registrados en una situacion de stock agotado o alerta de stock. Puede indicar alta demanda, reposicion durante el dia, desfase temporal del stock o posibles inconsistencias de datos.

</details>

2. Crees que esta situacion siempre es negativa? Justifica tu respuesta.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

No siempre. Puede ser negativa si refleja perdida de ventas o mala planificacion, pero tambien puede indicar alta rotacion y demanda fuerte. Hay que revisar la definicion exacta de `stock` y el momento en que se registra.

</details>

3. Calcula cuantos casos de este tipo hay por tienda.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Ejemplo:

```python
df_sales_stock_alert.groupBy("store_id").count().orderBy(F.desc("count")).show()
```

</details>


## 17. Activos e inactivos mediante `left_semi` y `left_anti`

El dataset no dispone de `customer_id`, por lo que se adapta el patrón clásico de clientes activos/inactivos a pares `store_id` + `product_id`.

### Idea del análisis

- Un par tienda-producto se considera **activo** si aparece en los últimos 30 días del dataset.
- Se considera **inactivo** si no aparece en ese periodo.

Esto permite practicar dos joins muy relevantes en Spark:
- `left_semi`: devuelve las filas del lado izquierdo que sí tienen coincidencia.
- `left_anti`: devuelve las filas del lado izquierdo que no tienen coincidencia.


In [ ]:
max_date = retail_df.agg(F.max("date").alias("max_date")).collect()[0]["max_date"]

recent_df = retail_df.filter(F.col("date") >= F.date_sub(F.lit(max_date), 30))

store_product_pairs = retail_df.select("store_id", "product_id").distinct()
recent_pairs = recent_df.select("store_id", "product_id").distinct()

df_active_pairs = store_product_pairs.join(
    recent_pairs,
    ["store_id", "product_id"],
    "left_semi"
)

df_inactive_pairs = store_product_pairs.join(
    recent_pairs,
    ["store_id", "product_id"],
    "left_anti"
)

print("Pares tienda-producto activos:", df_active_pairs.count())
print("Pares tienda-producto inactivos:", df_inactive_pairs.count())

### Ejercicio propuesto

1. Explica la diferencia entre `left_semi`, `left_anti` e `inner join`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

`inner join` devuelve columnas de ambas tablas para filas coincidentes. `left_semi` devuelve solo filas de la izquierda que tienen coincidencia en la derecha. `left_anti` devuelve solo filas de la izquierda que no tienen coincidencia en la derecha.

</details>

2. Que aplicaciones tendria este patron en un entorno retail?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Sirve para detectar productos vendidos o no vendidos, tiendas con o sin actividad, clientes que compraron frente a los que no compraron, o referencias activas sin stock. Es util para segmentacion y control de calidad de datos.

</details>

3. Repite el analisis usando un umbral de 60 dias.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Cambia el filtro temporal del conjunto auxiliar a 60 dias. Por ejemplo:

```python
recent_products_60d = retail_df.filter(F.col("date") >= F.date_sub(F.current_date(), 60)).select("product_id").distinct()
```

Despues aplica de nuevo `left_semi` o `left_anti` segun el objetivo.

</details>


## 18. Join por rango temporal con campañas simuladas

El dataset incluye información promocional a nivel de fila, pero para practicar un `range join` se construye una tabla auxiliar de campañas por periodos.

Una fila de ventas quedará asociada a una campaña si su fecha está comprendida entre:
- la fecha de inicio,
- y la fecha de fin del periodo.

Este tipo de join es útil en escenarios de:
- campañas,
- periodos de validez,
- reglas activas,
- ventanas temporales de negocio.


In [ ]:
promo_periods = spark.createDataFrame([
    ("campaign_1", "2017-01-01", "2017-06-30"),
    ("campaign_2", "2017-07-01", "2018-06-30"),
    ("campaign_3", "2018-07-01", "2019-12-31")
], ["campaign_id", "start_date", "end_date"])

promo_periods = (
    promo_periods
    .withColumn("start_date", F.to_date("start_date"))
    .withColumn("end_date", F.to_date("end_date"))
)

cond = (
    (retail_df["date"] >= promo_periods["start_date"]) &
    (retail_df["date"] < promo_periods["end_date"])
)

retail_campaign = retail_df.join(F.broadcast(promo_periods), cond, "left")

retail_campaign.select(
    "store_id", "product_id", "date", "revenue", "campaign_id"
).show(20, truncate=False)

### Ejercicio propuesto

1. Explica por que este join no es un equi-join clasico.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

No es un equi-join clasico porque no se une solo por igualdad de claves. Tambien incluye una condicion de rango temporal, por ejemplo que `date` este entre `start_date` y `end_date` de una campana.

</details>

2. Que problemas podrian aparecer si los periodos se solapan?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Una misma venta podria coincidir con varias campanas, generando duplicados y sobreestimacion de ingresos. Seria necesario definir prioridad de campanas, desduplicar o permitir multiatribucion de forma controlada.

</details>

3. Anade una agregacion que calcule la facturacion total por `campaign_id`.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Ejemplo:

```python
kpi_campaign = (
    campaign_sales_df
    .groupBy("campaign_id")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

kpi_campaign.show()
```

</details>


## 19. KPIs finales

En este bloque se construyen varios indicadores finales que resumen el comportamiento del dataset desde distintas perspectivas:

- por jerarquía principal de producto,
- por tamaño de tienda,
- por combinación de promociones.

Este tipo de salidas puede utilizarse como base para las conclusiones  o como tablas auxiliares para gráficos.


In [ ]:
kpi_h1 = (
    retail_df
    .groupBy("hierarchy1_id")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_store_size = (
    retail_df
    .groupBy("store_size")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

kpi_promos = (
    retail_df
    .groupBy("promo_type_1", "promo_type_2")
    .agg(
        F.sum("sales").alias("sales_qty"),
        F.sum("revenue").alias("revenue_total")
    )
    .orderBy(F.desc("revenue_total"))
)

print("KPIs por hierarchy1_id")
kpi_h1.show(20, truncate=False)

print("KPIs por tamaño de tienda")
kpi_store_size.show(20, truncate=False)

print("KPIs por promociones")
kpi_promos.show(20, truncate=False)

### Ejercicio propuesto

1. Resume que dimension parece mas explicativa del comportamiento de ventas.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Depende de los resultados agregados, pero normalmente producto y tienda suelen explicar gran parte de la variacion: el producto recoge categoria, precio y demanda; la tienda recoge ubicacion, tamano y formato comercial.

</details>

2. Selecciona tres KPIs y redacta una breve interpretacion como si formara parte del TFG.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Ejemplo: `total_revenue` mide la contribucion economica de cada tienda o producto; `stockout_ratio` evalua problemas de disponibilidad; `avg_revenue_per_unit` aproxima el ingreso medio por unidad vendida. En conjunto permiten analizar rendimiento, disponibilidad y valor unitario.

</details>

3. Propon dos graficos para acompanar estos resultados.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Dos graficos adecuados son: un ranking de barras horizontales con los productos o tiendas de mayor facturacion, y un scatter plot que compare `stockout_ratio` frente a `total_revenue` para detectar productos de alta demanda con problemas de disponibilidad.

</details>


## 20. Guardado opcional en Parquet

En entornos reales es habitual persistir resultados intermedios o finales en formatos columnares como Parquet.

Frente a CSV, Parquet ofrece ventajas como:
- compresión,
- mejor rendimiento de lectura,
- y preservación de tipos de datos.


In [ ]:
retail_df.write.mode("overwrite").parquet("out/retail_df")
df_pivot_promo1.write.mode("overwrite").parquet("out/df_pivot_promo1")
df_top_h1.write.mode("overwrite").parquet("out/df_top_h1")
stockout_kpi.write.mode("overwrite").parquet("out/stockout_kpi")

### Ejercicio propuesto

1. Explica por que Parquet suele ser preferible a CSV en pipelines analiticos.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Parquet es columnar, conserva esquema y tipos, comprime mejor y permite leer solo las columnas necesarias. CSV es texto plano, no guarda tipos y suele requerir mas coste de parsing.

</details>

2. Investiga que efecto tiene el particionado en Parquet sobre el rendimiento.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

El particionado permite que Spark descarte carpetas completas mediante partition pruning cuando hay filtros sobre columnas particionadas. Mejora lecturas selectivas, pero demasiadas particiones pequenas pueden generar muchos ficheros y overhead.

</details>

3. Propon una estrategia de particionado razonable para este dataset.

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Una estrategia razonable seria particionar por una columna temporal de baja cardinalidad, como `year_month`, y opcionalmente por una dimension estable como `store_id` solo si las consultas filtran mucho por tienda. Evitaria particionar por `product_id` si genera demasiadas carpetas.

</details>


## 21. Cierre de la sesión de Spark
Al finalizar el análisis, es recomendable cerrar la sesión de Spark para liberar recursos.


In [ ]:
spark.stop()